# SpaceX Falcon 9 — Data Collection with the SpaceX REST API

**Objective:** request launch data from the SpaceX REST API, parse the JSON response, enrich each launch with booster, launch site, payload and core information, keep only Falcon 9 launches, and handle missing values.

> Runs in **JupyterLite**. If the live API is unreachable from the browser, the notebook falls back to the dataset published by IBM so the rest of the analysis still runs.

In [34]:
import piplite
await piplite.install(['numpy', 'pandas'])

In [35]:
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [36]:
from js import fetch
import io, json

async def get_text(url):
    resp = await fetch(url)
    if not resp.ok:
        raise Exception(f"HTTP {resp.status} for {url}")
    return await resp.text()

async def get_json(url):
    return json.loads(await get_text(url))

async def get_csv(url):
    return pd.read_csv(io.StringIO(await get_text(url)))

## Helper functions
Each function takes the launch dataframe and calls a specific API endpoint using the IDs stored in each launch.

In [37]:
BASE = "https://api.spacexdata.com/v4/"

async def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = await get_json(BASE + "rockets/" + str(x))
            BoosterVersion.append(response['name'])

async def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = await get_json(BASE + "launchpads/" + str(x))
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

async def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = await get_json(BASE + "payloads/" + load)
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

async def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = await get_json(BASE + "cores/" + core['core'])
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

## Task 1: Request and parse the SpaceX launch data
We use the static snapshot of the `/v4/launches/past` response provided by IBM so results are consistent with the course.

In [38]:
static_json_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
launches_json = await get_json(static_json_url)
print("Launches received:", len(launches_json))

data = pd.json_normalize(launches_json)
data.head()

Launches received: 107


,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landin

### Keep only the needed columns and clean multi-core / multi-payload launches

In [39]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# Remove rows with multiple cores (Falcon Heavy) and multiple payloads
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# Each list has a single element -> extract it
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Convert date_utc to date and restrict the period
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
print(data.shape)

(94, 7)


## Task 2: Enrich each launch with the API endpoints

In [40]:
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs, LandingPad = [], [], [], [], [], []
Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], []

api_ok = True
try:
    await getBoosterVersion(data)
    await getLaunchSite(data)
    await getPayloadData(data)
    await getCoreData(data)
except Exception as e:
    api_ok = False
    print("SpaceX API not reachable from this browser:", e)

if api_ok:
    launch_dict = {'FlightNumber': list(data['flight_number']),
                   'Date': list(data['date']),
                   'BoosterVersion': BoosterVersion,
                   'PayloadMass': PayloadMass,
                   'Orbit': Orbit,
                   'LaunchSite': LaunchSite,
                   'Outcome': Outcome,
                   'Flights': Flights,
                   'GridFins': GridFins,
                   'Reused': Reused,
                   'Legs': Legs,
                   'LandingPad': LandingPad,
                   'Block': Block,
                   'ReusedCount': ReusedCount,
                   'Serial': Serial,
                   'Longitude': Longitude,
                   'Latitude': Latitude}
    launch_data = pd.DataFrame(launch_dict)
    print("Built from the live API:", launch_data.shape)

SpaceX API not reachable from this browser: TypeError: Failed to fetch


## Task 3: Filter the dataframe to keep only Falcon 9 launches
(Fallback: if the API was blocked, load IBM's equivalent dataset of the API results before filtering.)

In [41]:
if api_ok:
    data_falcon9 = launch_data[launch_data['BoosterVersion'] != 'Falcon 1'].copy()
    data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
else:
    data_falcon9 = await get_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
    print("Loaded IBM dataset_part_1.csv (fallback)")

print(data_falcon9.shape)
data_falcon9.head()

Loaded IBM dataset_part_1.csv (fallback)
(90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## Task 4: Data wrangling — handle missing values

In [42]:
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

In [43]:
# Replace missing PayloadMass with the mean (LandingPad keeps NaN = no landing pad used)
payload_mean = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, payload_mean)
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

In [44]:
print("Falcon 9 launches:", data_falcon9.shape[0])
data_falcon9.to_csv('dataset_part_1.csv', index=False)

Falcon 9 launches: 90


## Author

Jhoselyn Vera